# Top-k MoE — 基础专家混合

源码导航：[core/ffn/moe_base.py](../../../core/ffn/moe_base.py) 中的 `TopKMoE` 与 `load_balancing_loss`。

Fedus et al. (2021) 在 *Switch Transformers* 中提出：不扩大单个 FFN，而是并行维护多个**专家 (expert)** FFN，通过一个**路由网络 (router)** 将每个 token 分配给最相关的 $k$ 个专家。MoE 使模型总参数量随专家数线性增长，但前向激活的参数量仅随 $k$ 增长，实现"稀疏激活"的大规模扩展。

### 1. 理论推导

#### 1.1 路由与专家选择

给定输入 $x \in \mathbb{R}^{B \times T \times D}$，展平为 $x_{\text{flat}} \in \mathbb{R}^{N \times D}$（$N = B \cdot T$）。

Router 计算每个 token 到各专家的概率：

$$p = \text{softmax}(x_{\text{flat}} W_r), \quad p \in \mathbb{R}^{N \times E}$$

对每个 token 取 top-$k$ 专家及其权重：

$$g_i, e_i = \text{topk}(p_i), \quad i = 1, \ldots, N$$

输出为该 token 在所有选中专家上的加权求和：

$$y_i = \sum_{j \in e_i} g_{ij} \cdot \text{Expert}_j(x_i)$$

#### 1.2 容量限制 (Capacity Factor)

为避免少数热门专家过载，每个专家最多处理 $C$ 个 token：

$$C = \left\lceil \frac{N \cdot \text{cap}}{E} \right\rceil$$

超过容量的 token 被截断丢弃（对应专家输出为 0）。

#### 1.3 负载均衡损失 (Aux Loss)

Switch Transformer 引入辅助损失鼓励均匀分配：

$$\mathcal{L}_{\text{aux}} = \alpha \cdot E \sum_{e=1}^{E} f_e \cdot P_e$$

其中 $f_e$ 为分配给专家 $e$ 的 token 比例，$P_e$ 为 router 对专家 $e$ 的平均概率。当分配完全均匀时 $f_e = P_e = \frac{1}{E}$，损失取最小值。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.ffn.moe_base import TopKMoE

### 2. 形状与接口验证

In [ ]:
torch.manual_seed(0)
moe = TopKMoE(
    n_embd=128,
    num_experts=8,
    top_k=2,
    d_ffn=256,
    capacity_factor=1.25,
    aux_loss_coef=0.01,
)

x = torch.randn(2, 16, 128)
y, aux_loss = moe(x)

print(f"输入形状:   {tuple(x.shape)}")
print(f"输出形状:   {tuple(y.shape)}")
print(f"aux_loss:   {aux_loss.item():.6f}")
assert x.shape == y.shape, "MoE 必须保持输入输出维度一致！"
assert aux_loss is not None and aux_loss.item() >= 0, "训练时应返回非负 aux_loss"

### 3. top-k=1 (Switch Transformer) vs top-k=2 (GShard)

In [ ]:
moe_k1 = TopKMoE(n_embd=128, num_experts=8, top_k=1, d_ffn=256, capacity_factor=None)
moe_k2 = TopKMoE(n_embd=128, num_experts=8, top_k=2, d_ffn=256, capacity_factor=None)

torch.manual_seed(42)
x = torch.randn(2, 32, 128)

y1, _ = moe_k1(x)
y2, _ = moe_k2(x)

print(f"k=1 输出均值: {y1.mean().item():.4f},  std: {y1.std().item():.4f}")
print(f"k=2 输出均值: {y2.mean().item():.4f},  std: {y2.std().item():.4f}")
assert not torch.allclose(y1, y2, atol=1e-3), "不同 k 值应产生不同输出"

### 4. 容量限制验证

In [ ]:
torch.manual_seed(0)
moe_cap = TopKMoE(
    n_embd=64, num_experts=4, top_k=1, d_ffn=128,
    capacity_factor=0.5,  # 故意设得很低，制造截断
)
moe_cap.train()

x = torch.randn(1, 64, 64)
y, aux = moe_cap(x)

print(f"seq_len=64, 4 experts, cap=0.5")
print(f"每专家容量: ceil(64 * 0.5 / 4) = 8")
print(f"输出形状: {tuple(y.shape)}")
print(f"aux_loss: {aux.item():.4f}")
"容量限制下，部分 token 被截断，对应专家输出为 0"

### 5. 参数量分析

In [ ]:
n_embd, d_ffn = 128, 256
num_experts_list = [4, 8, 16, 32]

header = "{:>8s} {:>14s} {:>18s} {:>8s}".format("Experts", "Total Params", "Activated Params", "Ratio")
print(header)
print("-" * 55)
for ne in num_experts_list:
    moe = TopKMoE(n_embd=n_embd, num_experts=ne, top_k=2, d_ffn=d_ffn, capacity_factor=None)
    total = sum(p.numel() for p in moe.parameters())
    router_p = n_embd * ne
    expert_p = sum(p.numel() for p in moe.experts[0].parameters())
    activated = router_p + 2 * expert_p
    ratio = total / activated
    print(f"{ne:>8d} {total:>14,d} {activated:>18,d} {ratio:>8.1f}x")

"\n总参数量随专家数线性增长，但激活参数量仅随 top_k 固定。"

### 6. 源码精讲

```python
class TopKMoE(nn.Module):
    def __init__(self, n_embd, num_experts, top_k=2, d_ffn=256,
                 capacity_factor=1.25, aux_loss_coef=0.01, ...):
        super().__init__()
        self.router = nn.Linear(n_embd, num_experts, bias=False)
        self.experts = nn.ModuleList([... for _ in range(num_experts)])

    def forward(self, x):
        # 1. Router softmax
        router_logits = self.router(x_flat)              # (N, E)
        router_probs = F.softmax(router_logits, dim=-1)  # (N, E)

        # 2. Top-k selection + weight normalization
        top_k_weights, top_k_indices = torch.topk(router_probs, k)
        top_k_weights = top_k_weights / top_k_weights.sum(dim=-1, keepdim=True)

        # 3. Per-expert dispatch with capacity cap
        for e in range(num_experts):
            token_idx = (top_k_indices == e).nonzero()
            if len(token_idx) > capacity:
                token_idx = token_idx[:capacity]  # 截断
            out[token_idx] += w * expert_e(x[token_idx])

        # 4. Load-balancing loss
        aux_loss = num_experts * (f * P).sum() * coef
        return output, aux_loss
```

关键设计点：
- Router 不带 bias，避免引入先验偏差。
- top_k 权重归一化保证加权和的尺度稳定。
- 容量截断仅在 `self.training == True` 时生效，推理时通常设 `capacity_factor=None`。
- aux loss 鼓励均匀分配，系数 $\alpha$ 通常取 0.01，不主导主损失。

---

## 延伸阅读与参考资料

### 核心论文
- **Switch Transformers**: Fedus et al., 2021. [arXiv:2101.03961](https://arxiv.org/abs/2101.03961)
- **GShard**: Lepikhin et al., 2020. [arXiv:2006.16668](https://arxiv.org/abs/2006.16668)

### 工程实践
- **Mixtral 8x7B**: 8 个专家，top-k=2，激活参数量约 13B（总参数量 47B）。
- **Tutel (Microsoft)**: 高度优化的 MoE 训练系统，支持动态容量与专家并行。